# Домашнее задание 4: PWM and Markov Models

In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from Bio import SeqIO, motifs
from Bio.Seq import Seq
import warnings
warnings.filterwarnings('ignore')

BACKGROUND = {'A': 0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}
NUCLEOTIDES = ['A', 'C', 'G', 'T']
BG_PROBS = np.array([BACKGROUND[n] for n in NUCLEOTIDES])

sites = [
    "GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA",
    "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC",
    "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC",
    "CAGGTTGGC"
]
L = len(sites[0])
N = len(sites)

## Задание 2. Сканирование хромосомы человека

In [3]:
CHR1_PATH = 'chr1.fa'

import urllib.request, gzip, shutil
urllib.request.urlretrieve(
    'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr1.fa.gz',
    'chr1.fa.gz'
)
with gzip.open('chr1.fa.gz', 'rb') as f_in, open(CHR1_PATH, 'wb') as f_out:
    shutil.copyfileobj(f_in, f_out)

In [4]:
bio_sites = [Seq(s) for s in sites]
m = motifs.create(bio_sites)
m.background = BACKGROUND

record = next(SeqIO.parse(CHR1_PATH, 'fasta'))
sequence = record.seq[:1_000_000]

print(f'Мотив: {m.consensus}')
print(f'Загружено {len(sequence):,} нуклеотидов')

Мотив: CAGGTCAGC
Загружено 1,000,000 нуклеотидов


Транскрипционные факторы могут связываться на обеих цепях ДНК, поэтому ищем и на прямой, и на обратно-комплементарной.

In [5]:
THRESHOLD = 5.0
pssm = m.pssm
hits = []

for pos, score in pssm.search(sequence, threshold=THRESHOLD):
    hits.append((pos, '+', round(score, 4)))

for pos, score in pssm.reverse_complement().search(sequence, threshold=THRESHOLD):
    hits.append((pos, '-', round(score, 4)))

hits.sort(key=lambda x: x[0])

print(f'Найдено хитов (score > {THRESHOLD}): {len(hits)}')
print('\nПервые 20:')
for h in hits[:20]:
    print(f'  pos={h[0]:>8},  strand={h[1]},  score={h[2]:.4f}')

Найдено хитов (score > 5.0): 3084

Первые 20:
  pos= -989500,  strand=+,  score=12.2002
  pos= -988998,  strand=+,  score=8.4430
  pos= -988679,  strand=+,  score=5.5051
  pos= -988132,  strand=+,  score=7.5650
  pos= -988124,  strand=-,  score=8.6654
  pos= -988109,  strand=+,  score=7.0901
  pos= -987988,  strand=+,  score=6.0901
  pos= -987776,  strand=-,  score=5.0901
  pos= -987324,  strand=+,  score=11.5029
  pos= -986866,  strand=+,  score=9.3928
  pos= -986670,  strand=-,  score=7.0901
  pos= -986540,  strand=-,  score=6.8078
  pos= -986133,  strand=-,  score=6.9800
  pos= -985944,  strand=-,  score=5.4527
  pos= -985759,  strand=+,  score=8.1403
  pos= -985720,  strand=-,  score=15.7252
  pos= -984852,  strand=-,  score=8.6152
  pos= -983992,  strand=+,  score=5.5051
  pos= -983885,  strand=+,  score=6.0901
  pos= -983149,  strand=+,  score=6.2827
